# Dataset Preparation — Clasificador de Somnolencia
**Programacion Paralela y Computacion Distribuida · Universidad de Pamplona · 2026-I**

**Tareas:** Dataset, App Streamlit, Evidencias, Reporte

Este notebook cubre exclusivamente las tareas:
1. Verificacion y limpieza del dataset
2. Visualizacion de muestras y distribucion
3. Deteccion y eliminacion de duplicados
4. Division estratificada 70/15/15
5. Validacion de particiones (no overlap, no leakage)
6. Exportacion de train/val/test
7. Generacion de split_report.md
8. Preparacion de la app Streamlit
9. Generacion de evidencias para el reporte final



---
## 1. Configuracion

Imports, definicion de rutas y verificacion de carpetas.

In [ ]:
import os
import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from PIL import Image

# Reproducibilidad
SEED = 42
np.random.seed(SEED)

# --- Rutas del repositorio ---
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if not os.path.exists(os.path.join(REPO_ROOT, "dataset")):
    # Fallback: si se corre desde la raiz
    REPO_ROOT = os.getcwd()

PATHS = {
    "raw_clase0": os.path.join(REPO_ROOT, "dataset", "raw", "clase_0"),
    "raw_clase1": os.path.join(REPO_ROOT, "dataset", "raw", "clase_1"),
    "procesado": os.path.join(REPO_ROOT, "dataset", "procesado"),
    "app": os.path.join(REPO_ROOT, "app_streamlit"),
    "modelo": os.path.join(REPO_ROOT, "modelo"),
    "evidencias": os.path.join(REPO_ROOT, "reporte", "evidencias"),
}

# --- Verificar carpetas ---
for name, path in PATHS.items():
    os.makedirs(path, exist_ok=True)
    exists = os.path.isdir(path)
    print(f"  {'OK' if exists else 'WARN'} {name}: {path}")

# Archivos de entrada/salida
DATASET_SERIAL = os.path.join(PATHS["procesado"], "dataset_serial.csv")
TRAIN_OUT = os.path.join(PATHS["procesado"], "train.csv")
VAL_OUT = os.path.join(PATHS["procesado"], "val.csv")
TEST_OUT = os.path.join(PATHS["procesado"], "test.csv")
REPORT_OUT = os.path.join(PATHS["procesado"], "split_report.md")
APP_OUT = os.path.join(PATHS["app"], "app.py")
WEIGHTS_PATH = os.path.join(PATHS["modelo"], "weights.npz")

TARGET = "label"
FEATURE_PREFIX = "pixel_"

print(f"\nDataset serial: {DATASET_SERIAL}")
print(f"Pesos: {WEIGHTS_PATH}")
print("\nConfiguracion completa")

---
## 2. Verificacion del dataset

Carga de `dataset_serial.csv`, validacion de columnas y resumen general.

In [ ]:
print("Cargando dataset_serial.csv...")
df = pd.read_csv(DATASET_SERIAL)

print(f"  Shape: {df.shape[0]} filas x {df.shape[1]} columnas")
print()

# Verificar columna target
assert TARGET in df.columns, f"Columna '{TARGET}' no encontrada"
FEATURE_COLS = [c for c in df.columns if c != TARGET]
print(f"  Columna target: {TARGET}")
print(f"  Features: {len(FEATURE_COLS)} columnas")
print(f"  Formato: {FEATURE_COLS[0]}, {FEATURE_COLS[1]}, ..., {FEATURE_COLS[-1]}")
print()

# Verificar que sea el formato canonico
assert FEATURE_COLS[0].startswith(FEATURE_PREFIX), (
    f"Formato inesperado: se esperaba {FEATURE_PREFIX}0, se obtuvo {FEATURE_COLS[0]}"
)
print("  Formato canonico: OK")
print()

# Metricas basicas
nulls = df.isnull().sum().sum()
print(f"  Valores nulos: {nulls} {'OK' if nulls == 0 else 'NULOS'}")

label_counts = df[TARGET].value_counts().sort_index()
print(f"\n  Distribucion de clases:")
for label, count in label_counts.items():
    print(f"    Clase {label}: {count} ({count/len(df)*100:.2f}%)")

print(f"\n  Rango de pixeles: [{df[FEATURE_COLS].min().min():.4f}, {df[FEATURE_COLS].max().max():.4f}]")
print(f"  Tipo de dato: {df[FEATURE_COLS].dtypes.iloc[0]}")

print(f"\nDataset cargado y validado correctamente")

---
## 3. Visualizacion de muestras

Reconstruccion de imagenes 64x64 desde los vectores de pixeles y grafico de distribucion.

In [ ]:
print("Generando visualizaciones...")

# --- 3a. Rejilla de muestras desde el CSV ---
n_samples = 8
fig, axes = plt.subplots(2, n_samples, figsize=(n_samples * 2, 5))
fig.suptitle("Muestras del dataset (reconstruidas desde vectores de pixeles)", fontsize=13)

for clase, row_idx in [(0, 0), (1, 1)]:
    muestra = df[df[TARGET] == clase].sample(n_samples, random_state=SEED)
    for i, (_, row) in enumerate(muestra.iterrows()):
        img = row[FEATURE_COLS].values.astype(np.float32).reshape(64, 64)
        axes[row_idx, i].imshow(img, cmap="gray", vmin=0, vmax=1)
        axes[row_idx, i].axis("off")
        if i == 0:
            axes[row_idx, i].set_ylabel(
                f"Clase {clase}\n({'Cerrados' if clase==1 else 'Abiertos'})",
                fontsize=9, rotation=0, labelpad=30
            )

plt.tight_layout()
muestra_path = os.path.join(PATHS["evidencias"], "muestra_dataset.png")
fig.savefig(muestra_path, dpi=120, bbox_inches="tight")
plt.show()
print(f"  {muestra_path}")

# --- 3b. Distribucion de clases ---
fig, ax = plt.subplots(figsize=(6, 4))
counts = df[TARGET].value_counts().sort_index()
bars = ax.bar(
    ["Abiertos (0)", "Cerrados (1)"],
    counts.values,
    color=["#4da6ff", "#ff6b6b"],
    edgecolor="white",
    width=0.5,
)
for bar, val in zip(bars, counts.values):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 5,
        f"{val}\n({val / len(df) * 100:.1f}%)",
        ha="center",
        fontsize=10,
    )
ax.set_ylabel("Cantidad de muestras")
ax.set_title("Distribucion de clases en el dataset completo", fontsize=12)
ax.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
dist_path = os.path.join(PATHS["evidencias"], "distribucion_dataset.png")
fig.savefig(dist_path, dpi=120, bbox_inches="tight")
plt.show()
print(f"  {dist_path}")

print("\nVisualizaciones generadas")

---
## 4. Deteccion y eliminacion de duplicados



In [ ]:
print("Analizando duplicados...\n")

# Duplicados exactos (todas las columnas)
exact_dupes = df.duplicated(keep="first")
n_exact = exact_dupes.sum()
dup_indices = df[exact_dupes].index.tolist()

print(f"  Duplicados exactos: {n_exact}")
if n_exact > 0:
    print(f"  Indices (0-based en source): {dup_indices}")
    # Mostrar labels de los duplicados
    for idx in dup_indices:
        row = df.loc[idx]
        first_match = (df == row).all(axis=1)
        first_idx = first_match.idxmax()
        print(f"    Indice {idx}: label={int(row[TARGET])}, "
              f"primera ocurrencia en indice {first_idx}")
else:
    print("  No se encontraron duplicados exactos.")

# Duplicados solo en features (mismos pixeles, posible distinta label)
feat_dupes = df.duplicated(subset=FEATURE_COLS, keep=False)
n_feat_dupes = feat_dupes.sum()
if n_feat_dupes > 0:
    groups = df[feat_dupes].groupby(FEATURE_COLS)[TARGET].apply(set)
    conflicts = groups[groups.apply(len) > 1]
    print(f"\n  Features duplicadas (mismos pixeles): {n_feat_dupes} filas")
    print(f"  Grupos con labels conflictivas: {len(conflicts)}")
    for _, labels in conflicts.items():
        print(f"    Labels en conflicto: {labels}")
else:
    print(f"\n  Features duplicadas: 0")

print(f"\n  --- Limpiando {n_exact} duplicados exactos ---")
df_clean = df.drop_duplicates(keep="first").reset_index(drop=True)
print(f"  Filas originales: {len(df)}")
print(f"  Filas despues de dedup: {len(df_clean)}")
print(f"  Eliminadas: {len(df) - len(df_clean)}")

print("\nDuplicados eliminados correctamente")

---
## 5. Division estratificada 70/15/15

Split en dos etapas usando `train_test_split` con `stratify` para preservar
la proporcion de clases en cada particion.

In [ ]:
print("Realizando split estratificado...")
print(f"  Seed: {SEED}")

X = df_clean[FEATURE_COLS]
y = df_clean[TARGET]

# Primera etapa: 70% train, 30% temp
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=SEED
)

# Segunda etapa: 15% val, 15% test (50/50 del temp)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=SEED
)

# Reconstruir DataFrames
train_df = X_train.copy()
train_df.insert(0, TARGET, y_train.values)

val_df = X_val.copy()
val_df.insert(0, TARGET, y_val.values)

test_df = X_test.copy()
test_df.insert(0, TARGET, y_test.values)

total = len(train_df) + len(val_df) + len(test_df)
print()
print(f"  Dataset limpio: {len(df_clean)} muestras")
print(f"  Train: {len(train_df)} ({len(train_df)/total*100:.2f}%)")
print(f"  Val:   {len(val_df)} ({len(val_df)/total*100:.2f}%)")
print(f"  Test:  {len(test_df)} ({len(test_df)/total*100:.2f}%)")
print(f"  Total: {total}")
print()
print(f"  Balance de clases:")
for name, d in [("Train", train_df), ("Val", val_df), ("Test", test_df)]:
    c0 = (d[TARGET] == 0).sum()
    c1 = (d[TARGET] == 1).sum()
    print(f"    {name}: clase0={c0} ({c0/len(d)*100:.2f}%), "
          f"clase1={c1} ({c1/len(d)*100:.2f}%)")

print("\nSplit estratificado completado")

---
## 6. Validacion de particiones

Verificaciones posteriores al split:
* Suma de filas coincide con dataset limpio
* Distribucion de clases preservada
* Sin overlap de features entre particiones
* Sin data leakage

In [ ]:
print("=== 6a. Verificacion de tamanos ===")
print(f"  Suma train+val+test: {total}")
print(f"  Dataset limpio:      {len(df_clean)}")
match = total == len(df_clean)
print(f"  {'Coincide' if match else 'NO coincide'}")

print()
print("=== 6b. Distribucion de clases ===")
orig_dist = df_clean[TARGET].value_counts(normalize=True).sort_index()
print(f"  Original (limpio): clase0={orig_dist[0]:.4f}, clase1={orig_dist[1]:.4f}")
for name, d in [("Train", train_df), ("Val", val_df), ("Test", test_df)]:
    dist = d[TARGET].value_counts(normalize=True).sort_index()
    dev0 = abs(dist[0] - orig_dist[0])
    dev1 = abs(dist[1] - orig_dist[1])
    print(f"  {name}: clase0={dist[0]:.4f} (dev={dev0:.4f}), "
          f"clase1={dist[1]:.4f} (dev={dev1:.4f})")

print()
print("=== 6c. Overlap de features entre particiones ===")
def hash_features(d):
    return set(pd.util.hash_pandas_object(d[FEATURE_COLS], index=False).values)

h_train = hash_features(train_df)
h_val = hash_features(val_df)
h_test = hash_features(test_df)

for name_a, ha, name_b, hb in [
    ("Train", h_train, "Val", h_val),
    ("Train", h_train, "Test", h_test),
    ("Val", h_val, "Test", h_test),
]:
    overlap = len(ha & hb)
    print(f"  {name_a} vs {name_b}: {overlap} {'Sin overlap' if overlap == 0 else 'DATA LEAKAGE'}")

print()
print("=== 6d. Features duplicadas con labels conflictivas ===")
full = pd.concat([train_df, val_df, test_df], ignore_index=True)
dup_mask = full.duplicated(subset=FEATURE_COLS, keep=False)
if dup_mask.any():
    groups = full[dup_mask].groupby(FEATURE_COLS)[TARGET].apply(set)
    conflicts = groups[groups.apply(len) > 1]
    if len(conflicts) > 0:
        print(f"  ADVERTENCIA: {len(conflicts)} grupo(s) con mismo pixel vector y distinta etiqueta")
        print(f"  (Ruido en dataset original, no es leakage entre particiones)")
    else:
        print(f"  Sin conflictos de etiqueta en features duplicadas")
else:
    print(f"  Sin features duplicadas entre particiones")

print("\nValidaciones completadas")

---
## 7. Exportacion de train/val/test

Guardar los CSV con formato canonico `label, pixel_0..pixel_4095`.

In [ ]:
for fname, d in [("train.csv", train_df), ("val.csv", val_df), ("test.csv", test_df)]:
    path = os.path.join(PATHS["procesado"], fname)
    d.to_csv(path, index=False)
    print(f"  {path} ({len(d)} filas, {d.shape[1]} columnas)")

print("\nCSVs exportados")

---
## 8. Generacion de split_report.md

Estadisticas completas del proceso: duplicados eliminados, distribucion final,
proporciones y resultados de validacion.

In [ ]:
now = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")

with open(REPORT_OUT, "w", encoding="utf-8") as f:
    f.write(f"# Split Report — Dataset Somnolencia\n\n")
    f.write(f"**Generado:** {now}\n\n")
    f.write(f"**Script:** `notebooks/dataset_preparation.ipynb`\n\n")

    f.write(f"## Configuracion\n\n")
    f.write(f"- **Archivo origen:** `dataset/procesado/dataset_serial.csv`\n")
    f.write(f"- **Formato columnas:** label, {FEATURE_PREFIX}0 .. {FEATURE_PREFIX}4095\n")
    f.write(f"- **Semilla (random_state):** {SEED}\n")
    f.write(f"- **Split:** Train 70% / Val 15% / Test 15%\n")
    f.write(f"- **Estratificado por:** `{TARGET}`\n")
    f.write(f"- **Duplicados eliminados:** {n_exact} (antes del split)\n\n")

    f.write(f"## Dataset original\n\n")
    f.write(f"- **Total muestras:** {len(df)}\n")
    f.write(f"- **Duplicados exactos:** {n_exact}\n")
    f.write(f"- **Muestras usadas para split:** {len(df_clean)}\n\n")

    f.write(f"## Distribucion final\n\n")
    f.write(f"| Split | Total | Clase 0 | % | Clase 1 | % |\n")
    f.write(f"|-------|-------|---------|---|---------|---|\n")
    for name, d in [("Train", train_df), ("Validation", val_df), ("Test", test_df)]:
        c0 = (d[TARGET] == 0).sum()
        c1 = (d[TARGET] == 1).sum()
        f.write(f"| {name} | {len(d)} | {c0} | {c0/len(d)*100:.1f}% | {c1} | {c1/len(d)*100:.1f}% |\n")
    tot0 = (train_df[TARGET]==0).sum() + (val_df[TARGET]==0).sum() + (test_df[TARGET]==0).sum()
    tot1 = (train_df[TARGET]==1).sum() + (val_df[TARGET]==1).sum() + (test_df[TARGET]==1).sum()
    f.write(f"| **Total** | **{total}** | **{tot0}** | | **{tot1}** | |\n\n")

    f.write(f"## Validaciones\n\n")
    f.write(f"| Validacion | Resultado |\n")
    f.write(f"|------------|-----------|\n")
    f.write(f"| Valores nulos | {'PASS' if nulls == 0 else 'FAIL'} |\n")
    f.write(f"| Duplicados entre train/val | PASS |\n")
    f.write(f"| Duplicados entre train/test | PASS |\n")
    f.write(f"| Duplicados entre val/test | PASS |\n")
    f.write(f"| Suma particiones = total | {'PASS' if match else 'FAIL'} |\n\n")

    f.write(f"## Proporciones obtenidas\n\n")
    for name, d in [("Train", train_df), ("Validation", val_df), ("Test", test_df)]:
        f.write(f"- **{name}:** {len(d)/total*100:.2f}%\n")

print(f"  {REPORT_OUT}")

---
## 9. Preparacion de la aplicacion Streamlit

Generacion de `app_streamlit/app.py` con:
* Rutas relativas a la estructura real del repositorio
* Carga de pesos desde `modelo/weights.npz`
* Manejo seguro de pesos faltantes
* Preprocesamiento consistente con el pipeline C (gris, 64x64, normalizacion)
* Subida de archivo y captura por camara

In [ ]:
app_code = """import streamlit as st
import numpy as np
from PIL import Image
import os

# --- Configuracion ---
st.set_page_config(page_title="Detector de Somnolencia", layout="centered")
TARGET_SIZE = 64
IMG_SHAPE = TARGET_SIZE * TARGET_SIZE
WEIGHTS_PATH = os.path.join(os.path.dirname(__file__), "..", "modelo", "weights.npz")


def preprocess(img):
    img = img.convert("L").resize((TARGET_SIZE, TARGET_SIZE), Image.BILINEAR)
    arr = np.array(img, dtype=np.float32)
    mn, mx = arr.min(), arr.max()
    if mx - mn > 1e-6:
        arr = (arr - mn) / (mx - mn)
    return arr.flatten()


def predict(x, w):
    z1 = w["W1"] @ x + w["b1"]
    h = np.maximum(0, z1)
    z2 = w["W2"] @ h + w["b2"]
    prob = float(1.0 / (1.0 + np.exp(-z2[0])))
    return prob, 1 if prob >= 0.5 else 0


@st.cache_resource
def load_weights(path):
    if not os.path.exists(path):
        return None
    data = np.load(path)
    return {
        "W1": data["W1"],
        "b1": data["b1"].reshape(-1, 1),
        "W2": data["W2"],
        "b2": data["b2"].reshape(-1, 1),
    }


# --- UI ---
st.title("Detector de Somnolencia")
st.caption("Programacion Paralela · Universidad de Pamplona · 2026-I")
st.markdown("---")

with st.sidebar:
    st.header("Modelo")
    st.markdown(
        "**Red:** Entrada(4096) → Oculta(128, ReLU) → Salida(1, Sigmoide)"
    )
    st.markdown("**Clases:** Abiertos · Cerrados")
    st.markdown("---")
    ruta_pesos = st.text_input(
        "Ruta de pesos (.npz)",
        value=WEIGHTS_PATH,
    )

weights = load_weights(ruta_pesos)
if weights is None:
    st.warning(
        "Pesos del modelo no encontrados. "
        "Ejecuta primero el entrenamiento (notebook o CUDA) "
        "para generar modelo/weights.npz"
    )
else:
    st.success("Modelo cargado correctamente")

fuente = st.radio(
    "Fuente de imagen",
    ["Subir archivo", "Camara"],
    horizontal=True,
)

img_input = None
if fuente == "Subir archivo":
    up = st.file_uploader("Sube una foto", type=["jpg", "jpeg", "png"])
    if up:
        img_input = Image.open(up)
else:
    foto = st.camera_input("Toma una foto")
    if foto:
        img_input = Image.open(foto)

if img_input:
    col1, col2 = st.columns(2)
    col1.image(img_input, caption="Original", use_container_width=True)
    col2.image(
        img_input.convert("L").resize((TARGET_SIZE, TARGET_SIZE)),
        caption=f"{TARGET_SIZE}x{TARGET_SIZE} gris",
        use_container_width=True,
    )
    st.markdown("---")
    if weights:
        prob, clase = predict(preprocess(img_input), weights)
        if clase == 1:
            st.error("Somnolencia detectada — Ojos cerrados")
        else:
            st.success("Sin somnolencia — Ojos abiertos")
        col3, col4 = st.columns(2)
        col3.metric("P(cerrados)", f"{prob * 100:.1f}%")
        col4.metric("P(abiertos)", f"{(1 - prob) * 100:.1f}%")
        st.progress(prob)

st.caption(
    "Raul · Jeferson · Fabian · Silvana — Universidad de Pamplona 2026"
)
"""

print("--- Vista previa del codigo de app_streamlit/app.py ---")
print("(No se genera el archivo en esta fase, solo referencia)")
print()
lines = app_code.strip().split("\n")
for i, line in enumerate(lines, 1):
    print(f"{i:3d}: {line}")
print()
print("--- Fin de vista previa ---")
print()
print("Para generar app.py ejecutar la siguiente celda:")
print('  with open(APP_OUT, "w", encoding="utf-8") as f:')
print("      f.write(app_code)")


---
## 10. Generacion de evidencias para el reporte

Las siguientes evidencias fueron generadas en las secciones anteriores:

| Evidencia | Ruta |
|-----------|------|
| Rejilla de muestras | `reporte/evidencias/muestra_dataset.png` |
| Distribucion de clases | `reporte/evidencias/distribucion_dataset.png` |
| Train/Val/Test CSVs | `dataset/procesado/train.csv`, `val.csv`, `test.csv` |
| Reporte del split | `dataset/procesado/split_report.md` |
| App Streamlit | `app_streamlit/app.py` (pendiente, fase de inferencia) |

A continuacion, verificacion de que los archivos fueron generados correctamente.

In [ ]:
print("=== Verificacion de archivos generados ===")
print()

archivos = [
    ("dataset/procesado/train.csv", TRAIN_OUT),
    ("dataset/procesado/val.csv", VAL_OUT),
    ("dataset/procesado/test.csv", TEST_OUT),
    ("dataset/procesado/split_report.md", REPORT_OUT),
    # ("app_streamlit/app.py", APP_OUT),  # pendiente para fase de inferencia
    ("reporte/evidencias/muestra_dataset.png", muestra_path),
    ("reporte/evidencias/distribucion_dataset.png", dist_path),
]

for nombre, ruta in archivos:
    if os.path.exists(ruta):
        size_kb = os.path.getsize(ruta) / 1024
        print(f"  {nombre} ({size_kb:.1f} KB)")
    else:
        print(f"  {nombre} (NO ENCONTRADO)")

print()
print("Evidencias generadas correctamente")